# Card Transactions: fraud SQL starter

A small, fully synthetic credit-card dataset for practising fraud and analytics SQL: four tables (`accounts`, `merchants`, `transactions`, `chargebacks`) with fraud patterns planted on top of ordinary activity. Everything is generated from a fixed seed; no real person, card or merchant is in it. Licence: CC0 1.0 (public domain).

This notebook loads the four CSVs into an in-memory SQLite database and works through six fraud questions in plain SQL. Nothing to install: it uses only pandas, numpy and Python's built-in `sqlite3`.

The same questions also run graded in the browser at [sqlquest.app/fraud-analytics-sql](https://sqlquest.app/fraud-analytics-sql/).

## Load the data

The notebook looks for the Kaggle input folder first and falls back to the folder the notebook sits in, so it runs unchanged on Kaggle and locally.

In [1]:
import math
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

TABLES = ["accounts", "merchants", "transactions", "chargebacks"]

def find_data_dir():
    candidates = [Path("/kaggle/input/card-transactions-synthetic-fraud-sql-practice")]
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():  # the mount path can differ; find accounts.csv wherever it landed
        candidates += [p.parent for p in kaggle_root.rglob("accounts.csv")]
    candidates.append(Path.cwd())  # local: the folder next to the notebook
    for d in candidates:
        if all((d / f"{t}.csv").exists() for t in TABLES):
            return d
    raise FileNotFoundError("Could not find the four CSVs; add the dataset to this notebook.")

DATA_DIR = find_data_dir()
print("Reading from:", DATA_DIR if str(DATA_DIR).startswith("/kaggle") else "the folder next to this notebook")

dfs = {t: pd.read_csv(DATA_DIR / f"{t}.csv") for t in TABLES}
# related_chargeback_id is mostly empty; keep it an integer column, not a float
dfs["chargebacks"]["related_chargeback_id"] = dfs["chargebacks"]["related_chargeback_id"].astype("Int64")

con = sqlite3.connect(":memory:")
for name, df in dfs.items():
    df.to_sql(name, con, index=False)

# SQLite's math functions (sqrt, ...) are a compile-time option; register sqrt if this build lacks it
try:
    con.execute("SELECT sqrt(4)")
except sqlite3.OperationalError:
    con.create_function("sqrt", 1, math.sqrt, deterministic=True)

def q(sql):
    """Run a SQL query against the in-memory database and return a DataFrame."""
    return pd.read_sql_query(sql, con)

print("SQLite", sqlite3.sqlite_version, "| pandas", pd.__version__, "| numpy", np.__version__)
q("""
SELECT 'accounts' AS table_name, COUNT(*) AS n_rows FROM accounts
UNION ALL SELECT 'merchants', COUNT(*) FROM merchants
UNION ALL SELECT 'transactions', COUNT(*) FROM transactions
UNION ALL SELECT 'chargebacks', COUNT(*) FROM chargebacks
""")

Reading from: the folder next to this notebook
SQLite 3.46.0 | pandas 2.2.2 | numpy 1.26.4


,table_name,n_rows
0,accounts,200
1,merchants,25
2,transactions,2165
3,chargebacks,76


Joins: `transactions.account_id → accounts.account_id`, `transactions.merchant_id → merchants.merchant_id`, `chargebacks.txn_id → transactions.txn_id`, and `chargebacks.related_chargeback_id → chargebacks.chargeback_id`. Timestamps are ISO-8601 strings in UTC, which SQLite's date functions read directly.

In [2]:
q("SELECT * FROM transactions LIMIT 5")

,txn_id,account_id,amount,txn_at,merchant_id,lat,lng,status
0,1,149,151.84,2026-03-04T12:39:39.078Z,24,40.3420,-74.4092,completed
1,2,21,10.72,2026-03-04T13:04:50.641Z,7,52.9366,13.3229,completed
2,3,35,158.78,2026-03-04T13:27:15.528Z,25,52.7397,13.3134,completed
3,4,137,10.76,2026-03-04T13:49:08.426Z,14,41.3078,28.9955,completed
4,5,180,9.42,2026-03-04T15:08:40.041Z,20,40.5894,-73.7041,completed


## 1. Chargeback rate by merchant risk tier

What share of transactions ends in a chargeback, for each merchant risk tier? The `LEFT JOIN` keeps transactions without a chargeback in the denominator; each transaction has at most one chargeback here, and `COUNT(DISTINCT ...)` keeps the rate honest if that ever changes.

In [3]:
q("""
SELECT
    m.risk_tier,
    COUNT(DISTINCT t.txn_id)                                      AS transactions,
    COUNT(DISTINCT c.txn_id)                                      AS charged_back,
    ROUND(100.0 * COUNT(DISTINCT c.txn_id) / COUNT(DISTINCT t.txn_id), 2) AS chargeback_rate_pct
FROM transactions t
JOIN merchants m        ON m.merchant_id = t.merchant_id
LEFT JOIN chargebacks c ON c.txn_id = t.txn_id
GROUP BY m.risk_tier
ORDER BY CASE m.risk_tier WHEN 'high' THEN 1 WHEN 'medium' THEN 2 ELSE 3 END
""")

,risk_tier,transactions,charged_back,chargeback_rate_pct
0,high,262,10,3.82
1,medium,621,22,3.54
2,low,1282,44,3.43


The rate rises with the tier (high 3.82%, medium 3.54%, low 3.43%), but the whole spread is under half a percentage point: on this data, risk tier alone barely separates chargebacks.

## 2. Amount outliers: more than 3 standard deviations above the mean

SQLite has no `STDDEV`, so the CTE builds it from the identity variance = mean of squares − square of the mean (population standard deviation).

In [4]:
outliers = q("""
WITH stats AS (
    SELECT
        AVG(amount)                                   AS mean_amt,
        sqrt(AVG(amount * amount) - AVG(amount) * AVG(amount)) AS sd_amt
    FROM transactions
)
SELECT
    t.txn_id,
    t.account_id,
    t.amount,
    ROUND(s.mean_amt + 3 * s.sd_amt, 2)          AS threshold,
    ROUND((t.amount - s.mean_amt) / s.sd_amt, 1) AS z_score
FROM transactions t
CROSS JOIN stats s
WHERE t.amount > s.mean_amt + 3 * s.sd_amt
ORDER BY t.amount DESC
""")
print(len(outliers), "transactions above mean + 3 SD")
outliers

15 transactions above mean + 3 SD


,txn_id,account_id,amount,threshold,z_score
0,1984,42,9425.39,1593.51,19.5
1,2129,150,8754.25,1593.51,18.1
2,1878,140,7713.10,1593.51,15.9
3,1843,200,7672.64,1593.51,15.8
4,2059,17,6536.83,1593.51,13.4
5,1877,30,6312.91,1593.51,12.9
6,2089,156,6134.98,1593.51,12.5
7,1774,105,4877.63,1593.51,9.9
8,1772,75,4231.07,1593.51,8.5
9,1683,109,3792.71,1593.51,7.6


15 transactions clear the threshold of 1,593.51, each on a different account, with z-scores from 5.0 to 19.5. Keep the account ids in mind: several come back in the exercises below.

## 3. Velocity bursts: 6 or more transactions inside 5 minutes

For every transaction, a window frame counts the same account's transactions from that moment to 300 seconds later (`RANGE` over the Unix-epoch second). An account's worst window is the maximum of that count.

In [5]:
q("""
WITH txn AS (
    SELECT account_id, txn_id, txn_at,
           CAST(strftime('%s', txn_at) AS INTEGER) AS ts
    FROM transactions
),
windowed AS (
    SELECT account_id, txn_at,
           COUNT(*) OVER (
               PARTITION BY account_id
               ORDER BY ts
               RANGE BETWEEN CURRENT ROW AND 300 FOLLOWING
           ) AS txns_in_5_min
    FROM txn
)
SELECT
    w.account_id,
    a.status,
    MAX(w.txns_in_5_min) AS max_txns_in_5_min,
    MIN(w.txn_at) FILTER (WHERE w.txns_in_5_min >= 6) AS first_burst_starts
FROM windowed w
JOIN accounts a ON a.account_id = w.account_id
GROUP BY w.account_id, a.status
HAVING MAX(w.txns_in_5_min) >= 6
ORDER BY w.account_id
""")

,account_id,status,max_txns_in_5_min,first_burst_starts
0,17,active,6,2026-04-12T14:31:17.000Z
1,88,active,6,2026-04-26T01:23:39.000Z
2,109,active,6,2026-04-02T12:41:08.000Z
3,188,active,6,2026-04-10T11:57:16.000Z


Four accounts (17, 88, 109, 188) each reach exactly 6 transactions inside five minutes, and all four are `active`: none of them is flagged.

## 4. The shared-device ring

Which device fingerprints are used by more than one account, and who are those accounts?

In [6]:
q("""
WITH shared AS (
    SELECT device_fingerprint
    FROM accounts
    GROUP BY device_fingerprint
    HAVING COUNT(*) > 1
)
SELECT
    a.device_fingerprint,
    COUNT(*) OVER (PARTITION BY a.device_fingerprint) AS accounts_on_device,
    a.account_id,
    a.country,
    a.signup_at,
    a.status
FROM accounts a
JOIN shared s ON s.device_fingerprint = a.device_fingerprint
ORDER BY a.device_fingerprint, a.account_id
""")

,device_fingerprint,accounts_on_device,account_id,country,signup_at,status
0,dev_x4f2a9b1c7e,5,42,DE,2026-04-18T00:00:00.000Z,flagged
1,dev_x4f2a9b1c7e,5,88,JP,2026-02-10T00:00:00.000Z,active
2,dev_x4f2a9b1c7e,5,134,US,2026-04-14T00:00:00.000Z,active
3,dev_x4f2a9b1c7e,5,156,TR,2026-03-29T00:00:00.000Z,flagged
4,dev_x4f2a9b1c7e,5,195,TR,2026-02-22T00:00:00.000Z,flagged


One fingerprint, `dev_x4f2a9b1c7e`, is shared by 5 accounts in four countries; 3 of them are flagged and 2 (88 and 134) are still active. Account 88 is also one of the velocity-burst accounts from exercise 3.

## 5. Impossible travel

Pair each transaction with the same account's previous one (`LAG`), measure the great-circle distance between them, and divide by the time between them. SQLite has no trigonometry-based distance, so a haversine function written in Python is registered with `create_function` and called from SQL like a built-in.

Two thresholds: an implied speed above 1,000 km/h (faster than a commercial jet) and a jump of at least 1,000 km. The distance floor keeps the rule on cross-country jumps; without it, the short hops inside the velocity bursts from exercise 3 also clear the speed bar, because a few dozen kilometres in under a minute is fast too.

In [7]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance in km between two points given in degrees."""
    if None in (lat1, lng1, lat2, lng2):
        return None
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lng2 - lng1)
    h = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371.0 * math.asin(math.sqrt(h))

con.create_function("haversine_km", 4, haversine_km, deterministic=True)

q("""
WITH paired AS (
    SELECT
        account_id,
        txn_id,
        txn_at,
        lat, lng,
        LAG(txn_id) OVER w AS prev_txn_id,
        LAG(txn_at) OVER w AS prev_txn_at,
        LAG(lat)    OVER w AS prev_lat,
        LAG(lng)    OVER w AS prev_lng
    FROM transactions
    WINDOW w AS (PARTITION BY account_id ORDER BY txn_at, txn_id)
),
measured AS (
    SELECT
        account_id, prev_txn_id, txn_id, prev_txn_at, txn_at,
        haversine_km(prev_lat, prev_lng, lat, lng)             AS km,
        (julianday(txn_at) - julianday(prev_txn_at)) * 24 * 60 AS minutes
    FROM paired
    WHERE prev_txn_id IS NOT NULL
)
SELECT
    m.account_id,
    a.status,
    m.prev_txn_id,
    m.txn_id,
    m.prev_txn_at,
    m.txn_at,
    ROUND(m.km)                        AS km,
    ROUND(m.minutes, 1)                AS minutes,
    ROUND(m.km / (m.minutes / 60.0))   AS implied_kmh
FROM measured m
JOIN accounts a ON a.account_id = m.account_id
WHERE m.km >= 1000
  AND (m.minutes = 0 OR m.km / (m.minutes / 60.0) > 1000)
ORDER BY implied_kmh DESC
""")

,account_id,status,prev_txn_id,txn_id,prev_txn_at,txn_at,km,minutes,implied_kmh
0,42,flagged,2060,2061,2026-04-30T12:00:00.000Z,2026-04-30T12:11:00.000Z,10844.0,11.0,59148.0
1,195,flagged,1720,1723,2026-04-21T12:00:00.000Z,2026-04-21T12:12:00.000Z,11423.0,12.0,57116.0
2,156,flagged,1617,1618,2026-04-18T12:00:00.000Z,2026-04-18T12:24:00.000Z,16996.0,24.0,42491.0


Three pairs, 10,844 to 16,996 km apart in 11 to 24 minutes. All three are on flagged accounts (42, 195, 156), the same three flagged accounts that share the device in exercise 4.

## 6. The chargeback chain

`related_chargeback_id` points a chargeback at an earlier one it belongs with. A recursive CTE starts from the roots (chargebacks that point nowhere but are pointed at) and walks down, carrying the depth and the path.

In [8]:
q("""
WITH RECURSIVE chain AS (
    SELECT
        c.chargeback_id,
        c.chargeback_id                 AS root_id,
        0                               AS depth,
        CAST(c.chargeback_id AS TEXT)   AS path
    FROM chargebacks c
    WHERE c.related_chargeback_id IS NULL
      AND EXISTS (SELECT 1 FROM chargebacks k WHERE k.related_chargeback_id = c.chargeback_id)
    UNION ALL
    SELECT
        k.chargeback_id,
        chain.root_id,
        chain.depth + 1,
        chain.path || ' -> ' || k.chargeback_id
    FROM chargebacks k
    JOIN chain ON k.related_chargeback_id = chain.chargeback_id
)
SELECT
    chain.root_id,
    chain.depth,
    chain.path,
    cb.account_id,
    cb.reason_code,
    cb.opened_at,
    cb.status
FROM chain
JOIN chargebacks cb ON cb.chargeback_id = chain.chargeback_id
ORDER BY chain.root_id, chain.path
""")

,root_id,depth,path,account_id,reason_code,opened_at,status
0,1,0,1,160,service_not_received,2026-03-06T16:52:17.429Z,split
1,1,1,1 -> 2,42,fraud_card_not_present,2026-03-16T23:50:46.792Z,open
2,1,2,1 -> 2 -> 3,48,fraud_card_present,2026-03-12T05:22:57.195Z,cardholder_won
3,4,0,4,180,service_not_received,2026-03-21T11:43:48.754Z,merchant_won
4,4,1,4 -> 5,89,fraud_card_present,2026-03-21T02:16:21.506Z,split
5,4,1,4 -> 7,53,duplicate_charge,2026-03-21T03:08:00.403Z,cardholder_won


Six chargebacks in two trees: a three-deep chain 1 → 2 → 3 and a parent 4 with two children, 5 and 7. Each tree spans different accounts, and in both a child was opened before the chargeback it points to.

## Try next

Three more questions on the same four tables, no answers given:

1. Average transaction amount by account country, computed two ways: over all transactions, and as the average of each account's own average. Where do the two disagree, and why?
2. Which accounts share an `ip_block`? Do any of them overlap with the shared-device ring from exercise 4?
3. For each merchant category, what share of chargebacks was won by the cardholder, and how many chargebacks are still open?